In [1]:
import pandas  as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split

from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import torch

In [2]:
SEED = 42
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
NUM_CLASSES = 3

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

df = pd.read_csv('dataset_filtered.csv')
df['use_no_bg'] = df['use_no_bg'].astype(bool)

label_to_idx = {l:i for i,l in enumerate(sorted(df['label'].unique()))}
idx_to_label = {l:i for l, i in label_to_idx.items()}
class_names = sorted(label_to_idx.keys())

cuda


In [3]:
print(f"Total sampel: {len(df)}")
print(f"Label mapping: {label_to_idx}")

Total sampel: 26197
Label mapping: {'Electronic': 0, 'Organic': 1, 'Recyclable': 2}


## Pipeline

In [4]:
class MixedDataset(Dataset):
    """
    Dataset bakal memilih acak gambar yang di-augmentasi berikut:
    - Gambar asli + augmentasi (sedikit lebih agresif)
    - Gambar no-background + augmentasi (jika use_no_bg=True)
    """
    def __init__(self, dataframe, heavy_transform, light_transform):
        self.df = dataframe.reset_index(drop=True)
        self.heavy = heavy_transform
        self.light = light_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = label_to_idx[row['label']]

        if row['use_no_bg'] and random.random() < 0.5:
            img = Image.open(row['final_path']).convert('RGB')
            img = self.light(img)
        else:
            img = Image.open(row['file_path']).convert('RGB')
            img = self.heavy(img)
        return img, label

In [5]:
class EvalDataset(Dataset):
    """
    Dataset untuk validasi (nanti di split)
    """
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['file_path']).convert('RGB')
        img = self.transform(img)
        return img, label_to_idx[row['label']]

In [6]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc="[Train]")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return running_loss / len(loader)

In [7]:
def validasi_epoch(model, loader, criterion, device):
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="[Val]"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return val_loss / len(loader), acc, all_preds, all_labels

In [10]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs, device, class_names):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{num_epochs}")

        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, preds, labels = validasi_epoch(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        scheduler.step()

    return history, preds, labels

In [11]:
def plot_hasil(history, all_preds, all_labels, class_names):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss curves
    axes[0].plot(history['train_loss'], marker='o', label='Train Loss')
    axes[0].plot(history['val_loss'], marker='o', label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Curves')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy curve
    axes[1].plot(history['val_acc'], marker='o', color='green', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Validation Accuracy')
    axes[1].grid(True, alpha=0.3)

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[2])
    axes[2].set_xlabel('Predicted')
    axes[2].set_ylabel('True')
    axes[2].set_title('Confusion Matrix')

    plt.tight_layout()
    plt.show()

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

In [13]:
def model(model_name='resnet50', num_classes=3):
    """
    Factory model untuk sekali inisiasi

    :param model_name: Nama model yang akan digunakan (pilih antara resnet50, efficientnet_b3, convnext_tiny, swin_t). Default: resnet50
    :param num_classes: Jumlah label/class
    :return: model
    """

    match model_name:
        case 'resnet50':
            model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
            model.fc = nn.Linear(model.fc.in_features, num_classes)
        case 'efficientnet_b3':
            model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
            model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        case 'convnext_tiny':
            model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
            model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
        case 'swin_t':
            model = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
            model.head = nn.Linear(model.head.in_features, num_classes)
    return model

In [14]:
# buat heavy augment
htrain = transforms.Compose([
    transforms.Resize((256)),
    transforms.RandomResizedCrop(224, scale=(0.08,1.0), ratio=(0.75,1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# light augment
ltrain = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.3,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# eval
etransform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [16]:
full_dataset = MixedDataset(df, htrain, ltrain)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])


df_val = df.iloc[val_dataset.indices]
val_eval_dataset = EvalDataset(df_val, etransform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,drop_last=True)
val_loader = DataLoader(val_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [17]:
train_labels = [label_to_idx[df.iloc[i]['label']] for i in train_dataset.indices]
class_counts = np.bincount(train_labels)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float).to(device)
class_weights = class_weights / class_weights.sum() * len(class_counts)
print(f"Class weights: {class_weights.cpu().numpy()}")

Class weights: [1.7646334  0.5433843  0.69198227]


In [ ]:
# class FocalLoss(nn.Module):
#     def __init__(self, alpha=None, gamma=2.0):
#         super().__init__()
#         self.alpha = alpha
#         self.gamma = gamma
#
#     def forward(self, inputs, targets):
#         ce_loss = nn.CrossEntropyLoss(weight=self.alpha, reduction='none')
#         ce = ce_loss(inputs, targets)
#         pt = torch.exp(-ce)
#         focal_loss = ((1 - pt) ** self.gamma * ce).mean()
#         return focal_loss

In [18]:
mixup_fn = Mixup(
    mixup_alpha=1.0,
    cutmix_alpha=1.0,
    prob=0.5,
    switch_prob=0.5,
    mode='batch',
    num_classes=NUM_CLASSES
)

## Resnet50

In [ ]:
# model_resnet = model('resnet50', num_classes=3).to(device)

In [ ]:
# criterion = nn.CrossEntropyLoss(weight=class_weights)
# criterion_soft = SoftTargetCrossEntropy()
# optimizer = optim.AdamW(model_resnet.parameters(), lr=LEARNING_RATE)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# train_losses = []
# val_losses = []
# val_accuracies = []
#
# for epoch in range(EPOCHS):
#     # Training
#     model_resnet.train()
#     running_loss = 0.0
#     pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
#     for imgs, labels in pbar:
#         imgs, labels = imgs.to(device), labels.to(device)
#         imgs, labels_mixed = mixup_fn(imgs, labels)
#
#         optimizer.zero_grad()
#         outputs = model_resnet(imgs)
#
#         if labels_mixed.dim() == 1:
#             loss = criterion(outputs, labels_mixed)
#         else:
#             loss = criterion_soft(outputs, labels_mixed)
#
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()
#         pbar.set_postfix(loss=f"{loss.item():.4f}")
#
#     train_losses.append(running_loss / len(train_loader))
#
#     # Validasi
#     model_resnet.eval()
#     val_loss = 0.0
#     all_preds, all_labels = [], []
#     with torch.no_grad():
#         for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
#             imgs, labels = imgs.to(device), labels.to(device)
#             outputs = model_resnet(imgs)
#             loss = criterion(outputs, labels)
#             val_loss += loss.item()
#             _, preds = torch.max(outputs, 1)
#             all_preds.extend(preds.cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())
#
#     val_losses.append(val_loss / len(val_loader))
#     acc = np.mean(np.array(all_preds) == np.array(all_labels))
#     val_accuracies.append(acc)
#     print(f"Epoch {epoch+1}: Train Loss={train_losses[-1]:.4f}, Val Loss={val_losses[-1]:.4f}, Val Acc={acc:.4f}")
#     scheduler.step()

In [ ]:
# history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_acc': val_accuracies}
# plot_hasil(history, all_preds, all_labels, class_names)

## Efficientnet_B3

In [ ]:
# model_b3 = model('efficientnet_b3', num_classes=3).to(device)

In [ ]:
# criterion = nn.CrossEntropyLoss(weight=class_weights)
# criterion_soft = SoftTargetCrossEntropy()
# optimizer = optim.AdamW(model_b3.parameters(), lr=LEARNING_RATE)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# train_losses = []
# val_losses = []
# val_accuracies = []
#
# for epoch in range(EPOCHS):
#     # Training
#     model_b3.train()
#     running_loss = 0.0
#     pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
#     for imgs, labels in pbar:
#         imgs, labels = imgs.to(device), labels.to(device)
#         imgs, labels_mixed = mixup_fn(imgs, labels)
#
#         optimizer.zero_grad()
#         outputs = model_b3(imgs)
#
#         if labels_mixed.dim() == 1:
#             loss = criterion(outputs, labels_mixed)
#         else:
#             loss = criterion_soft(outputs, labels_mixed)
#
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()
#         pbar.set_postfix(loss=f"{loss.item():.4f}")
#
#     train_losses.append(running_loss / len(train_loader))
#
#     # Validasi
#     model_b3.eval()
#     val_loss = 0.0
#     all_preds, all_labels = [], []
#     with torch.no_grad():
#         for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
#             imgs, labels = imgs.to(device), labels.to(device)
#             outputs = model_b3(imgs)
#             loss = criterion(outputs, labels)
#             val_loss += loss.item()
#             _, preds = torch.max(outputs, 1)
#             all_preds.extend(preds.cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())
#
#     val_losses.append(val_loss / len(val_loader))
#     acc = np.mean(np.array(all_preds) == np.array(all_labels))
#     val_accuracies.append(acc)
#     print(f"Epoch {epoch+1}: Train Loss={train_losses[-1]:.4f}, Val Loss={val_losses[-1]:.4f}, Val Acc={acc:.4f}")
#     scheduler.step()

In [ ]:
# history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_acc': val_accuracies}
# plot_hasil(history, all_preds, all_labels, class_names)

In [ ]:
# history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_acc': val_accuracies}
# plot_hasil(history, all_preds, all_labels, class_names)

## ConvNext

In [ ]:
model_convn = model('convnext_tiny', num_classes=3).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
criterion_soft = SoftTargetCrossEntropy()
optimizer = optim.AdamW(model_convn.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(EPOCHS):
    # Training
    model_convn.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        imgs, labels_mixed = mixup_fn(imgs, labels)

        optimizer.zero_grad()
        outputs = model_convn(imgs)

        if labels_mixed.dim() == 1:
            loss = criterion(outputs, labels_mixed)
        else:
            loss = criterion_soft(outputs, labels_mixed)

        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_losses.append(running_loss / len(train_loader))

    # Validasi
    model_convn.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model_convn(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_losses.append(val_loss / len(val_loader))
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    val_accuracies.append(acc)
    print(f"Epoch {epoch+1}: Train Loss={train_losses[-1]:.4f}, Val Loss={val_losses[-1]:.4f}, Val Acc={acc:.4f}")
    scheduler.step()

In [ ]:
history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_acc': val_accuracies}
plot_hasil(history, all_preds, all_labels, class_names)

In [ ]:
torch.save(model_convn.state_dict(), 'trained_model/convnext2.pth')

## Swin T

In [19]:
model_swin = model('swin_t', num_classes=3).to(device)

In [21]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
criterion_soft = SoftTargetCrossEntropy()
optimizer = optim.AdamW(model_swin.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [22]:
N_FOLDS = 3
EPOCHS_CV = 10

y = df['label'].values
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*60}")

    df_train_fold = df.iloc[train_idx]
    df_val_fold = df.iloc[val_idx]

    train_dataset_fold = MixedDataset(df_train_fold, htrain, ltrain)
    val_dataset_fold = EvalDataset(df_val_fold, etransform)

    train_loader_fold = torch.utils.data.DataLoader(
        train_dataset_fold, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True
    )
    val_loader_fold = torch.utils.data.DataLoader(
        val_dataset_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    model_swin_fold = model('swin_t', num_classes=3).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model_swin_fold.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_CV)

    history, preds, labels = train_model(
        model_swin_fold, train_loader_fold, val_loader_fold, criterion, optimizer, scheduler,
        EPOCHS_CV, device, class_names
    )

    torch.save(model_swin_fold.state_dict(), f'trained_model/swin_cv_fold_{fold+1}.pth')

    f1_macro = f1_score(labels, preds, average='macro')
    fold_results.append({
        'fold': fold + 1,
        'macro_f1': f1_macro,
        'val_acc': history['val_acc'][-1],
        'val_loss': history['val_loss'][-1]
    })




FOLD 1/3

Epoch 1/10


[Val]: 100%|██████████| 273/273 [01:13<00:00,  3.70it/s]


Train Loss: 0.3120 | Val Loss: 0.1763 | Val Acc: 0.9407

Epoch 2/10


[Val]: 100%|██████████| 273/273 [01:11<00:00,  3.83it/s]


Train Loss: 0.2331 | Val Loss: 0.1631 | Val Acc: 0.9414

Epoch 3/10


...

Train Loss: 0.0996 | Val Loss: 0.1237 | Val Acc: 0.9614

Epoch 9/10


[Val]: 100%|██████████| 273/273 [00:59<00:00,  4.60it/s]


Train Loss: 0.0910 | Val Loss: 0.1181 | Val Acc: 0.9626

Epoch 10/10


[Val]: 100%|██████████| 273/273 [00:59<00:00,  4.57it/s]


Train Loss: 0.0831 | Val Loss: 0.1175 | Val Acc: 0.9621


In [24]:
for res in fold_results:
    print(f"Fold {res['fold']}: Macro F1 = {res['macro_f1']:.4f}, Acc = {res['val_acc']:.4f}")

macro_f1_scores = [res['macro_f1'] for res in fold_results]
print(f"\nRata-rata Macro F1: {np.mean(macro_f1_scores):.4f}")
print(f"Standar Deviasi: {np.std(macro_f1_scores):.4f}")

Fold 1: Macro F1 = 0.9702, Acc = 0.9678
Fold 2: Macro F1 = 0.9649, Acc = 0.9623
Fold 3: Macro F1 = 0.9645, Acc = 0.9621

Rata-rata Macro F1: 0.9666
Standar Deviasi: 0.0026


### Swin-CutMix

In [ ]:
# train_losses = []
# val_losses = []
# val_accuracies = []
#
# for epoch in range(EPOCHS):
#     # Training
#     model_swin.train()
#     running_loss = 0.0
#     pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
#     for imgs, labels in pbar:
#         imgs, labels = imgs.to(device), labels.to(device)
#         imgs, labels_mixed = mixup_fn(imgs, labels)
#
#         optimizer.zero_grad()
#         outputs = model_swin(imgs)
#
#         if labels_mixed.dim() == 1:
#             loss = criterion(outputs, labels_mixed)
#         else:
#             loss = criterion_soft(outputs, labels_mixed)
#
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()
#         pbar.set_postfix(loss=f"{loss.item():.4f}")
#
#     train_losses.append(running_loss / len(train_loader))
#
#     # Validasi
#     model_swin.eval()
#     val_loss = 0.0
#     all_preds, all_labels = [], []
#     with torch.no_grad():
#         for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
#             imgs, labels = imgs.to(device), labels.to(device)
#             outputs = model_swin(imgs)
#             loss = criterion(outputs, labels)
#             val_loss += loss.item()
#             _, preds = torch.max(outputs, 1)
#             all_preds.extend(preds.cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())
#
#     val_losses.append(val_loss / len(val_loader))
#     acc = np.mean(np.array(all_preds) == np.array(all_labels))
#     val_accuracies.append(acc)
#     print(f"Epoch {epoch+1}: Train Loss={train_losses[-1]:.4f}, Val Loss={val_losses[-1]:.4f}, Val Acc={acc:.4f}")
#     scheduler.step()

In [ ]:
# history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_acc': val_accuracies}
# plot_hasil(history, all_preds, all_labels, class_names)

In [ ]:
# model_swin.eval()

In [ ]:
# torch.save(model_swin.state_dict(), 'swin_t2.pth')

## Testing

In [ ]:
df_sub = pd.read_csv('submission.csv')
ids = df_sub['id'].values

df_sub.info()

In [ ]:
class TestDataset(Dataset):
    def __init__(self, ids, test_dir, transform=None):
        self.ids = ids
        self.test_dir = Path(test_dir)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_path = self.test_dir / f"{self.ids[idx]}.jpg"
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

In [ ]:
test_dataset = TestDataset(ids, 'test', etransform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

In [ ]:
# Salah mapping
# Model kita: 0=Electronic, 1=Organic, 2=Recyclable
# Ketentuan lomba: 0=Recyclable, 1=Electronic, 2=Organic
model_to_sub = {0: 1, 1: 2, 2: 0}

predicted_labels = []
with torch.no_grad():
    for imgs in tqdm(test_loader, desc="Inference Swin‑T"):
        imgs = imgs.to(device)
        outputs = model_swin(imgs)
        _, preds = torch.max(outputs, 1)
        predicted_labels.extend([model_to_sub[int(p)] for p in preds.cpu().numpy()])

In [ ]:
type(predicted_labels[4])

In [ ]:
# submission_df = pd.DataFrame({'id': ids, 'predicted': predicted_labels})
# submission_df.to_csv('submission_swin.csv', index=False)

In [ ]:
df_sub = pd.read_csv('submission_swin.csv')
df_sub.info()

In [ ]:
df_sub.head(10)